# Integrate Modern Data Architectures with Generative AI and Interact Using Prompts for Querying SQL Databases and APIs

This notebook demonstrates how large language models (LLMs) that are accessible through [Amazon Bedrock](https://aws.amazon.com/bedrock/), such as Amazon Nova, interact with AWS databases, data stores, and third-party data warehousing solutions, such as [Amazon Athena](https://aws.amazon.com/athena/features/). This interaction is showcased by generating and running SQL queries, and by making requests to API endpoints. The LangChain framework, used to accomplish this demonstration, allows an LLM to interact with its environment and connect with other sources of data. The LangChain framework operates based on three principles: calling out to a language model, being data-aware, and being agentic. 

This notebook focuses on establishing a connection to one data source, consolidating metadata, and returning fact-based data points in response to user queries by using LLMs and LangChain. The solution can be enhanced to add multiple data sources.


<img src='img-genai-sql-langchain-overall-solution.png' width="800" height="600">


### Prerequisites:
1. Use the Python 3 (Data Science 3.0) kernel.
2. Install the required packages.
3. Run the one-time setup by entering the user input parameters, copying the dataset, and running the AWS Glue crawler.
3. Access to the FM. The Amazon Nova Micro model is used in this notebook. For more information, see Amazon Nova models in the Amazon Bedrock User Guide at https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-nova.html.



### Solution walkthrough:

Step 1. Read into a pandas DataFrame the library data JSON file that was downloaded from Amazon S3.

Step 2. Populate the AWS Glue Data Catalog by running an AWS Glue crawler on the staged JSON file. 

Step 3. To obtain information on the data schema, use the SQLAlchemy library to query the Data Catalog.

Step 4. Define the functions to determine the best data channel to answer the user query, and then generate a response to the user query as a correctly formatted SQL statement.

Step 5. Prompt the LLM through LangChain to determine the data channel. After determining the data channel, run the Langchain SQL database chain to convert 'text to sql', and then run the query against the source data channel. 

Finally, display the results.

## Code cell 1 ##


In [ ]:
# NOTE: package installation moved to requirements.txt
# Run `pip install -r requirements.txt` in your virtualenv instead of installing inside the notebook
print('Dependencies should be installed from requirements.txt')

## Code cell 2 — Imports ##

In [ ]:
import os
import json
import time
import logging
import boto3
from botocore.exceptions import ClientError

from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd() / 'src'))

from llm_sql.runner import build_athena_service

### Library dataset
The dataset for this solution was originally stored in an Amazon Simple Storage Service (Amazon S3) bucket and downloaded locally as the file, s3_library_data.json. 

The dataset consists of the following columns:
* book_id
* title
* author
* genre
* pub_date

You can use the following commands to load the JSON data into a pandas DataFrame and view the data.

## Code cell 3 ##

In [ ]:
import pandas as pd
library_df = pd.read_json('s3_library_data.json', lines=True)
# Normalize and parse dates
library_df['pub_date'] = pd.to_datetime(library_df.get('pub_date'), errors='coerce')
print(library_df.info())
library_df.head()

## One-time setup
### AWS CloudFormation outputs

Some of the resources needed for this notebook have already been created as part of project deployment. These preprovisioned resources include S3 buckets to store data, AWS Glue databases, and AWS Glue crawlers. 

In order to proceed with the process, you first need to set some variables:
* Project files bucket
* Library database name
* Library crawler name

Note: Make sure to modify the following CloudFormation stack name.

## Code cell 4 ##


In [ ]:
## Update with the correct CloudFormation stack name.
cloudformation_stack_name = os.environ.get('CFN_STACK_NAME', '<<Replace_With_StackName>>')
if cloudformation_stack_name.startswith('<<'):
    raise EnvironmentError(
        'Set the CFN_STACK_NAME environment variable or replace the placeholder '
        'with your deployed CloudFormation stack name before running this notebook.'
    )

cfn_client = boto3.client('cloudformation')

def get_cfn_outputs(stack_name: str) -> dict:
    """Return a dict of {OutputKey: OutputValue} for the given stack."""
    try:
        stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
    except ClientError as exc:
        raise RuntimeError(f'Could not describe stack {stack_name!r}: {exc}') from exc
    if not stacks:
        raise RuntimeError(f'Stack {stack_name!r} not found.')
    return {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}

outputs = get_cfn_outputs(cloudformation_stack_name)
print(json.dumps(outputs, indent=2))

## Code cell 5 ##

In [ ]:
# The following variables define the AWS Glue Data Catalog database and crawler to use.

## Project note ##  You must update some of the following variables as part of the later Project section of this solution.
data_file = 's3_library_data.json'
project_files_folder = 'library-data'
glue_db_name = outputs['LibraryDatabaseName']
glue_crawler_name = outputs['LibraryCrawlerName']

# Primary project S3 bucket (eu-north-1) — created by CloudFormation
project_files_bucket = outputs['ProjectfilesBucketName']

print('Project files bucket:', project_files_bucket)

### Stage data

Use the following command to copy the library JSON data to the S3 bucket's library-data folder.

Note: In the later Project section, you must modify some of the previously initialized variables.

## Code cell 6 ##

In [ ]:
# Upload the data file to the project S3 bucket.
!aws s3 cp {data_file} s3://{project_files_bucket}/{project_files_folder}/

### Run the AWS Glue crawler

The crawler runs and crawls the specified S3 bucket and folder. The crawler then creates tables based on the data found. The crawled data is available for querying through Amazon Athena, or nontechnical users can use the power of LLMs to convert text questions into SQL.

## Code cell 7 ##

In [ ]:
glue_client = boto3.client('glue')

print('About to start running the crawler:', glue_crawler_name)

try:
    glue_client.start_crawler(Name=glue_crawler_name)
    print('Started crawler. Polling for completion...')
    timeout = 60 * 10  # 10 minutes
    interval = 10
    waited = 0
    while waited < timeout:
        crawler_info = glue_client.get_crawler(Name=glue_crawler_name)['Crawler']
        state = crawler_info.get('State')
        print(f"Crawler '{glue_crawler_name}' state: {state}")
        if state in ('READY', 'STOPPED'):
            last_crawl = crawler_info.get('LastCrawl', {})
            last_status = last_crawl.get('Status', 'UNKNOWN')
            if last_status == 'FAILED':
                error_msg = last_crawl.get('ErrorMessage', 'No error message available.')
                raise RuntimeError(f'Crawler failed: {error_msg}')
            print(f'Crawler finished successfully. Last crawl status: {last_status}')
            break
        time.sleep(interval)
        waited += interval
    else:
        raise TimeoutError(f'Crawler did not finish within {timeout}s.')

except ClientError as exc:
    logger.error('AWS error starting or polling crawler: %s', exc)
    raise
except (RuntimeError, TimeoutError) as exc:
    logger.error('%s', exc)
    raise

Note: Before proceeding to the next step, confirm that the crawler state returned to READY and that the last crawl status is not FAILED.

### Step 1 - Connect to databases by using SQLAlchemy. 

LangChain uses SQLAlchemy to connect to SQL databases. SQLDatabaseChain can be used with any SQL dialect supported by SQLAlchemy, such as MS SQL, MySQL, MariaDB, PostgreSQL, Oracle SQL, and SQLite. For more information about requirements for connecting to your database, see the SQLAlchemy documentation. 

Note: The following code establishes a database connection for data sources and LLMs. Note that the solution will work only if the database connection for your sources is defined in the next code cell. For more information, see the previous Prerequisites section.

## Code cell 8 ##

In [ ]:
# Use the reusable Athena + Bedrock runtime helper from src/llm_sql.
region = boto3.session.Session().region_name or 'eu-north-1'

athena_service = build_athena_service(
    glue_db_names=[glue_db_name],
    project_files_bucket=project_files_bucket,
    region=region,
    athena_workgroup=os.environ.get('ATHENA_WORKGROUP', 'primary'),
    bedrock_model=os.environ.get('BEDROCK_MODEL', 'amazon.nova-micro-v1:0'),
    secrets_manager_secret=os.environ.get('SECRETS_MANAGER_SECRET'),
)

dbathena = athena_service.db
glue_catalog = athena_service.glue_catalog
allowed_tables = athena_service.allowed_tables

print('Connection to Athena database succeeded for Glue DB:', glue_db_name)
print('Sample catalog entries:')
print('\n'.join(glue_catalog.splitlines()[:10]))

### Step 2 - Generate dynamic prompt templates.
Build a consolidated view of the AWS Glue Data Catalog by combining metadata stored for all the databases in pipe-delimited format.

## Code cell 9 ##

In [ ]:
# The reusable runtime service has already populated the Glue catalog and allowed tables.
# The notebook retains the same variable names for compatibility.

print('Sample catalog entries:\n', '\n'.join(glue_catalog.splitlines()[:10]))
print('\nAllowed tables (sample):', list(allowed_tables)[:10])

### Step 3 - Define two functions.

Define two functions that (1) determine the best data channel to answer the user query, and (2) generate a response to the user's query.

## Code cell 10 ##

In [ ]:
# Reuse the reusable runtime service from src/llm_sql.
llm_service = athena_service

def identify_channel(query: str):
    return llm_service.identify_channel(query)

## Code cell 11 ##

In [ ]:
def run_query(query: str) -> str:
    return llm_service.run_query(query)

### Step 4 - Run the run_query function.

Running the run_query function, in turn, calls the Langchain SQL database chain to convert 'text to sql', and then runs the query against the source data channel.

The following samples are provided for test runs. Uncomment one query at a time to run it.
## Code cell 12 ##

In [ ]:
# query = """How many books with a genre of Fantasy are in the library?""" 
# query = """Find 3 books in the library with Tarzan in the title?""" 
# query = """How many books by author Stephen King are in the library?""" 
query = """How many total books are there in the library?""" 

response =  run_query(query)
print("----------------------------------------------------------------------")
print(f'SQL and response from user query {query}  \n  {response}')

### Hands-On Project: Add the Cars Dataset

The steps below walk you through adding the Cars dataset as a second data source.
Complete these steps in order after the library pipeline is working.

**What to do:**

1. In code cell 5, change `data_file` to `'s3_cars_data_normalized.csv'`,
   `project_files_folder` to `'cars-data'`, and update `glue_db_name` and
   `glue_crawler_name` to use the Cars outputs from CloudFormation
   (`outputs['CarsDatabaseName']` and `outputs['CarsCrawlerName']`).

2. Re-run code cells 5 → 6 → 7 to upload the normalized CSV and run the Cars crawler.

3. Re-run code cell 8 to reconnect Athena to the Cars Glue database.

4. Re-run code cell 9 to refresh the catalog metadata.

5. Run the sample queries in the cell below.

**Tip:** Run `python scripts/normalize_cars.py` locally first to produce
`s3_cars_data_normalized.csv` before uploading.

## Code cell 13 — Cars queries ##



In [ ]:
# Cars dataset queries — uncomment one at a time and run.
# Prerequisite: complete the setup steps in the markdown cell above first.

# query = """What car has the highest horsepower?"""
# query = """What is the make and price of the cheapest car?"""
# query = """What is the average price of a car?"""
# query = """How many cars have more than 6 cylinders?"""
query = """What is the average price of a car?"""

response = run_query(query)
print('-' * 70)
print(f'Query : {query}')
print(f'Answer: {response}')

---
## Optional Integrations — Snowflake & Amazon Redshift

The cells below add **Snowflake** and **Amazon Redshift** as alternative data sources.
Everything is commented out. To activate a source:

1. Install the required packages (see the comment at the top of each cell).
2. Set the required environment variables (listed in each cell).
3. Uncomment the cell and run it.
4. Uncomment the matching entry in the **Multi-Source Registry** cell at the bottom.

The `run_query()` and `identify_channel()` functions defined earlier work unchanged —
they operate on whichever `SQLDatabase` instance is registered as the active source.

---

### Code cell 14 — Snowflake connection

**Required package** (add to `requirements.txt` and reinstall):
```
snowflake-sqlalchemy>=1.5.0,<2.0
```

**Required environment variables:**
| Variable | Example value |
|---|---|
| `SNOWFLAKE_USER` | `jane.doe` |
| `SNOWFLAKE_PASSWORD` | *(use Secrets Manager in production)* |
| `SNOWFLAKE_ACCOUNT` | `xy12345.eu-west-1` |
| `SNOWFLAKE_DATABASE` | `ANALYTICS` |
| `SNOWFLAKE_SCHEMA` | `PUBLIC` |
| `SNOWFLAKE_WAREHOUSE` | `COMPUTE_WH` |
| `SNOWFLAKE_ROLE` | `ANALYST` *(optional, defaults to PUBLIC)* |

**Tip:** On SageMaker or Lambda, retrieve credentials from AWS Secrets Manager:
```python
secret = boto3.client('secretsmanager').get_secret_value(SecretId='snowflake/prod')
creds  = json.loads(secret['SecretString'])
```

In [ ]:
# =============================================================================
# SNOWFLAKE CONNECTION  —  uncomment this entire cell to activate
# Prerequisite: pip install 'snowflake-sqlalchemy>=1.5.0,<2.0'
# =============================================================================

# import os
# from sqlalchemy import create_engine
# from langchain_community.utilities import SQLDatabase
#
# # --- Credentials (set as environment variables, never hardcode) ---
# _sf_user      = os.environ['SNOWFLAKE_USER']
# _sf_password  = os.environ['SNOWFLAKE_PASSWORD']
# _sf_account   = os.environ['SNOWFLAKE_ACCOUNT']    # e.g. xy12345.eu-west-1
# _sf_database  = os.environ['SNOWFLAKE_DATABASE']
# _sf_schema    = os.environ['SNOWFLAKE_SCHEMA']
# _sf_warehouse = os.environ['SNOWFLAKE_WAREHOUSE']
# _sf_role      = os.environ.get('SNOWFLAKE_ROLE', 'PUBLIC')
#
# # --- Build connection string ---
# _sf_conn_str = (
#     f'snowflake://{_sf_user}:{_sf_password}@{_sf_account}'
#     f'/{_sf_database}/{_sf_schema}'
#     f'?warehouse={_sf_warehouse}&role={_sf_role}'
# )
#
# # --- Create engine and SQLDatabase ---
# _sf_engine   = create_engine(_sf_conn_str)
# db_snowflake = SQLDatabase(_sf_engine)
#
# # --- Use SQLAlchemy introspection instead of Glue catalog ---
# sf_catalog        = db_snowflake.get_table_info()
# sf_allowed_tables = set(db_snowflake.get_usable_table_names())
#
# print('Snowflake connected. Tables:', list(sf_allowed_tables)[:10])

### Code cell 15 — Amazon Redshift connection

**Required packages** (add to `requirements.txt` and reinstall):
```
redshift-connector>=2.0.0,<3.0
sqlalchemy-redshift>=0.8.0,<1.0
```

**Two auth options are shown below — choose one:**

- **Option A — Password** (simpler, use Secrets Manager in production)
- **Option B — IAM** (recommended for AWS-hosted workloads; no password needed)

**Required environment variables (Option A):**
| Variable | Example value |
|---|---|
| `REDSHIFT_HOST` | `my-cluster.abc123.eu-north-1.redshift.amazonaws.com` |
| `REDSHIFT_PORT` | `5439` *(default)* |
| `REDSHIFT_DATABASE` | `dev` |
| `REDSHIFT_USER` | `awsuser` |
| `REDSHIFT_PASSWORD` | *(use Secrets Manager in production)* |

**Additional variable for Option B (IAM):**
| Variable | Example value |
|---|---|
| `REDSHIFT_CLUSTER_ID` | `my-cluster` *(omit for Serverless)* |

**Redshift Serverless host format:**
`<workgroup>.<account-id>.<region>.redshift-serverless.amazonaws.com`

In [ ]:
# =============================================================================
# AMAZON REDSHIFT CONNECTION  —  uncomment this entire cell to activate
# Prerequisite: pip install 'redshift-connector>=2.0.0,<3.0' 'sqlalchemy-redshift>=0.8.0,<1.0'
# =============================================================================

# import os
# from sqlalchemy import create_engine
# from langchain_community.utilities import SQLDatabase
#
# _rs_host     = os.environ['REDSHIFT_HOST']
# _rs_port     = os.environ.get('REDSHIFT_PORT', '5439')
# _rs_database = os.environ['REDSHIFT_DATABASE']
# _rs_user     = os.environ['REDSHIFT_USER']
#
# # ---------------------------------------------------------------------------
# # OPTION A — Password auth (simpler; store password in Secrets Manager)
# # ---------------------------------------------------------------------------
# # _rs_password = os.environ['REDSHIFT_PASSWORD']
# # _rs_conn_str = (
# #     f'redshift+redshift_connector://{_rs_user}:{_rs_password}'
# #     f'@{_rs_host}:{_rs_port}/{_rs_database}'
# # )
# # _rs_engine   = create_engine(_rs_conn_str)
#
# # ---------------------------------------------------------------------------
# # OPTION B — IAM auth (recommended; no password required)
# # The IAM role attached to this environment must have
# # redshift:GetClusterCredentials permission.
# # ---------------------------------------------------------------------------
# import redshift_connector
#
# def _make_redshift_conn():
#     return redshift_connector.connect(
#         iam=True,
#         host=_rs_host,
#         port=int(_rs_port),
#         database=_rs_database,
#         db_user=_rs_user,
#         cluster_identifier=os.environ.get('REDSHIFT_CLUSTER_ID', ''),
#         region=boto3.session.Session().region_name or 'eu-north-1',
#     )
#
# # Wrap the IAM connection in SQLAlchemy via a creator function
# # _rs_engine = create_engine(
# #     'redshift+redshift_connector://',
# #     creator=_make_redshift_conn,
# # )
#
# # --- Create SQLDatabase (works with either Option A or B engine) ---
# db_redshift = SQLDatabase(_rs_engine)
#
# # --- Use SQLAlchemy introspection instead of Glue catalog ---
# rs_catalog        = db_redshift.get_table_info()
# rs_allowed_tables = set(db_redshift.get_usable_table_names())
#
# print('Redshift connected. Tables:', list(rs_allowed_tables)[:10])

### Code cell 16 — Multi-source registry & query routing

Once one or more additional sources are connected (cells 14–15), register them here
and update the active source used by `run_query()`.

The registry maps a short source name to its `SQLDatabase` instance and catalog.
The Bedrock prompt in `identify_channel()` is extended to include a description of
each source so the model can route the query to the right database automatically.

**To switch active source:** change `ACTIVE_SOURCE` to `'snowflake'` or `'redshift'`.

In [ ]:
# =============================================================================
# MULTI-SOURCE REGISTRY  —  uncomment and extend as you add sources
# =============================================================================

# # --- Registry: map source name → (SQLDatabase, catalog_string, allowed_tables) ---
# DATA_SOURCES = {
#     'athena': {
#         'db':             dbathena,
#         'catalog':        glue_catalog,
#         'allowed_tables': allowed_tables,
#         'description':    'Library and Cars datasets stored in S3, queried via Athena.',
#     },
#     # Uncomment after running cell 14:
#     # 'snowflake': {
#     #     'db':             db_snowflake,
#     #     'catalog':        sf_catalog,
#     #     'allowed_tables': sf_allowed_tables,
#     #     'description':    'Enterprise data warehouse on Snowflake.',
#     # },
#     # Uncomment after running cell 15:
#     # 'redshift': {
#     #     'db':             db_redshift,
#     #     'catalog':        rs_catalog,
#     #     'allowed_tables': rs_allowed_tables,
#     #     'description':    'Analytics data warehouse on Amazon Redshift.',
#     # },
# }
#
# # --- Set the active source (change this to switch databases) ---
# ACTIVE_SOURCE = 'athena'   # options: 'athena' | 'snowflake' | 'redshift'
#
# _src = DATA_SOURCES[ACTIVE_SOURCE]
#
# # Override the globals used by identify_channel() and run_query()
# dbathena        = _src['db']           # reuse the same variable name so no other cell changes
# glue_catalog    = _src['catalog']
# allowed_tables  = _src['allowed_tables']
#
# print(f'Active source: {ACTIVE_SOURCE}')
# print(f'Tables available: {list(allowed_tables)[:10]}')